# Coding agent -- end to end

Runs a coding task through `coding_agent_app/main.py` the way Cloud Tasks will, without a deployed service: it writes a task with
`write_coding_agent_task`, posts to `POST /run/{task_id}` in-process, and reads the task back from Firestore while it runs.

Needs your own credentials (ADC) for Firestore and Secret Manager, and a ready sandbox snapshot. A task whose patch applies **pushes a real
`agent/<task_id>` branch** to the ML repo (with the `github-agent-push-token` secret); `aisle_recency_ambiguous` and `promo_share_ambiguous`
should stop with questions and push nothing, so they are the safe first runs.

In [1]:
import asyncio
import json
import sys
import time

import httpx

sys.path.insert(0, "..")

from IPython.display import Markdown, display

import coding_agent_app.main as service
from app.coding_task_store import get_coding_agent_task, write_coding_agent_task
from app.config import CODING_AGENT_DEFAULT_BASE_BRANCH, GITHUB_REPO_NAME, GITHUB_REPO_OWNER
from scripts.coding_agent_tasks import (
    aisle_pacing,
    aisle_pacing_lgbm_ranker,
    aisle_pacing_lgbm_ranker_w_test_notebook,
    aisle_pacing_all_models_edit_notebook,
    revision_for_aisle_pacing,
    column_descriptions,
    cumulative_weekday_counts,
    rapidfuzz_dependency,
    aisle_recency_ambiguous,
    promo_share_ambiguous,
)

CONVERSATION_ID = "coding-agent-e2e-test"  # standing in for a real conversation
USER_ID = "coding-agent-test-user"  # standing in for a real token's claims["oid"]

# timeout=None: a run lasts minutes
client = httpx.AsyncClient(transport=httpx.ASGITransport(app=service.app), base_url="http://coding-agent", timeout=None)

## Pick a task

Any task imported above. `REVIEW = True` also runs the grader. `BASE_BRANCH` set to an earlier task's branch makes the task a revision: it continues on that branch.

In [ ]:
SELECTED = promo_share_ambiguous
REVIEW = True
BASE_BRANCH = CODING_AGENT_DEFAULT_BASE_BRANCH
# for revision_for_aisle_pacing, the branch of the aisle_pacing run's result

plan = SELECTED.model_copy(update={"review": REVIEW, "base_branch": BASE_BRANCH})
print(f"{plan.name} | {len(plan.requirements)} requirements, {len(plan.constraints)} constraints, "
      f"{len(plan.decisions)} decisions, {len(plan.suggested_code)} suggested code blocks | review: {plan.review} | base branch: {plan.base_branch}")

Promo share of purchases | 2 requirements, 3 constraints, 0 decisions, 0 suggested code blocks | review: True | base branch: agent/aisle-pacing-ratio-fe0185


## Create the task

The same call `start_coding_task` will make. The task is written as `queued`.

In [11]:
task_id = await write_coding_agent_task(conversation_id=CONVERSATION_ID, user_id=USER_ID, plan=plan)
task = await get_coding_agent_task(task_id)
print("task_id:", task_id)
print("status:", task["status"], "| expires_at:", task["expires_at"])

task_id: 63c38d18-1fb1-426b-a10c-93f8b07d5066
status: queued | expires_at: 2026-12-09 22:27:40.637393+00:00


## Run it

Posts to `/run/{task_id}`, which returns when the run is done, and prints each change of the task's status and progress as it happens.
The first call loads the secrets and the push token.

In [ ]:
run = asyncio.create_task(client.post(f"/run/{task_id}"))
started = time.perf_counter()
last = None
try:
    while not run.done():
        task = await get_coding_agent_task(task_id)
        current = (task["status"], task["progress"])
        if current != last:
            last = current
            print(f"[{time.perf_counter() - started:>5.0f}s] {task['status']}: {task['progress']}")
        await asyncio.sleep(2)
except BaseException:  # the cell was interrupted: cancel the run, and with it the agent and its sandbox
    run.cancel()
    raise

response = run.result()
print(f"[{time.perf_counter() - started:>5.0f}s] POST /run returned {response.status_code} {response.json()}")

[    0s] queued: 


INFO coding_agent_app.orchestration: coding task 63c38d18-1fb1-426b-a10c-93f8b07d5066 started


[    2s] running: Starting the sandbox...
[   11s] running: The agent is working...


WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_share', path='/', glob=None)
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_pacing_ratio', path='/', glob='*.{sqlx,py,ipynb,md,json,yaml}')
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_pacing_ratio', path='/home/user/repo', glob=None)
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_share', path='/home/user/repo', glob=None)
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_share', path='/', glob=None)
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_pacing_ratio', path='/', glob=None)
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_pacing_ratio', path='/home/user/repo', glob=None)
WARNING deepagents.backends.sandbox: agrep timed out after 35s (pattern='user_aisle_share', path='/home/user/re

## Watch a task the deployed service runs

Instead of "Run it": create the task above, then start it from Cloud Tasks (a terminal command, see Step 21 of the setup guide). This cell only polls the task until it completes; the next cells show its result.

In [ ]:
started = time.perf_counter()
last = None
while True:
    task = await get_coding_agent_task(task_id)
    current = (task["status"], task["progress"])
    if current != last:
        last = current
        print(f"[{time.perf_counter() - started:>5.0f}s] {task['status']}: {task['progress']}")
    if task["status"] in ("completed", "notified"):
        break
    await asyncio.sleep(2)

## The result

What is stored for the main agent.

In [7]:
task = await get_coding_agent_task(task_id)
result = task["coding_agent_result"]
print(f"status: {task['status']} | outcome: {result['outcome']} | stop reason: {result['stop_reason']}")
print(f"started from: {result['started_from_commit']} | dev moved: {result['dev_moved']}")
if result["branch"]:
    print(f"branch: {result['branch']} | commit: {result['commit']}")
    print(f"compare: https://github.com/{GITHUB_REPO_OWNER}/{GITHUB_REPO_NAME}/compare/dev...{result['branch']}")

print("\nREPORT")
display(Markdown(result["report"] or "(no report)"))

for label, key in (("Assumptions", "assumptions"), ("Blocking questions", "blocking_questions"), ("Follow-up questions", "follow_up_questions"),
                   ("Patch problems", "patch_problems")):
    if result[key]:
        print(f"\n{label}:")
        for item in result[key]:
            print(f"  - {item}")

if result["files"]:
    print("\nFiles changed:")
    for file in result["files"]:
        print(f"  +{file['added']:<4} -{file['removed']:<4} {file['path']}")

if result["checks"]:
    print("\nChecks:")
    for check in result["checks"]:
        print(f"  {'PASS' if check['passed'] else 'FAIL'}  {check['name']}")
        if check["detail"]:
            print("        " + check["detail"][-400:].replace("\n", "\n        "))

review = result["review"]
if review["ran"]:
    print(f"\nReview: {review['verdict']} after {review['rounds']} rounds. {review['explanation']}")
    for criterion in review["criteria"]:
        if not criterion["passed"]:
            print(f"  FAIL  {criterion['criterion']} -> {criterion['gap']}")

if result["todos"]:
    print("\nTodos:")
    for todo in result["todos"]:
        print(f"  [{todo['status']}] {todo['content']}")

usage = result["token_usage"]
for label in ("agent", "reviewer"):
    counts = usage[label]
    if counts:
        print(f"\ntokens, {label}: {counts['input']:,} input ({counts['cache_read']:,} from cache, {counts['cache_write']:,} written) "
              f"and {counts['output']:,} output")

status: completed | outcome: succeeded | stop reason: None
started from: 72782f4b40d8f62aa26ae4b34d905d61808f720d | dev moved: False
branch: agent/aisle-pacing-ratio-fe0185 | commit: e56b9831fb4ccd5b1aaf66011a7c7706c4003fad
compare: https://github.com/Jason-Handerhan/Kaggle-Instacart-Reorder-Engine-Portfolio-Project/compare/dev...agent/aisle-pacing-ratio-fe0185

REPORT


I removed the duplicate `user_aisle_pacing_ratio` description from the `columns:` block of `final_ml_features_table.sqlx`. It now appears once. After that edit I recompiled the project and dry-ran the composed `final_ml_features_table` query in BigQuery. It passed and returns all 40 documented columns, including `user_aisle_pacing_ratio`. I did not run the pipeline, so the values and the new null-check assertion are untested.

The full change set, all under `definitions/`:
- **`silver/prelim_user_aisle_features.sqlx`**: three new documented columns. They are `user_aisle_first_purchase_day`, `user_aisle_last_purchase_day` and `user_aisle_cumulative_purchase_orders`.
- **`gold/skeleton_features.sqlx`**: the three columns are forward-filled by user and aisle and documented.
- **`gold/final_ml_features_table.sqlx`**: the `user_aisle_pacing_ratio` expression, its column description (now once), and the `user_aisle_pacing_ratio IS NOT NULL` row condition.

The silver and skeleton tables were dry-run earlier and I haven't edited them since.

The ratio is the label day minus the aisle's last purchase day, divided by the average gap. The average gap is the anchor order's `days_since_first_order` minus the first aisle purchase day, divided by (aisle purchase orders − 1). The average is 999 when undefined, and the ratio is 999 when it can't be computed.


Assumptions:
  - Where the average gap is undefined, it is set to 999 before the division, as the product version does. The ratio is then days since last purchase divided by 999, not 999.
  - The training feature list in `src/` or `config/` was left alone, as the task limits edits to `/definitions`.

Follow-up questions:
  - Should the training feature list in `src/` or `config/` include `user_aisle_pacing_ratio`? That is outside this task's scope.

Files changed:
  +12   -1    definitions/gold/final_ml_features_table.sqlx
  +6    -0    definitions/gold/skeleton_features.sqlx
  +22   -6    definitions/silver/prelim_user_aisle_features.sqlx

Checks:
  PASS  patch passes validation
  PASS  patch applies to a fresh clone of the current dev
  PASS  patch applies in the clean sandbox
  PASS  compile passes in the clean sandbox

Review: satisfied after 2 rounds. All the criteria that apply pass. The diff adds the silver columns, the skeleton forward-fill, the final `user_aisle_pacing_ratio`

## A second call does nothing

The task is no longer `queued`, so a repeated or forged call is refused without touching it.

In [8]:
again = await client.post(f"/run/{task_id}")
print(again.status_code, again.json())
assert again.status_code == 409

409 {'detail': 'Coding task fe018535-a9a0-4971-bffe-8395c93ad414 is completed, not queued, so it will not run again.'}


In [9]:
print(json.dumps(await get_coding_agent_task(task_id), indent=2, default=str))

{
  "conversation_id": "coding-agent-e2e-test",
  "progress": "Checking the patch in a clean sandbox...",
  "task_id": "fe018535-a9a0-4971-bffe-8395c93ad414",
  "pr_url": null,
  "created_at": "2026-11-09 22:21:40.690954+00:00",
  "completed_at": "2026-11-09 22:24:23.180311+00:00",
  "plan": {
    "goal": "Add a new ML feature, user_aisle_pacing_ratio, to the Dataform pipeline: (days since the user's last purchase in the aisle, as of the label order) divided by (the user's average days between purchases in that aisle). It is the aisle-level counterpart of user_product_nearing_reorder_ratio.\n\nExploratory analysis found that reorder rate rises and then declines as this ratio grows, and that the feature is not redundant with the model's existing top features. That analysis used a 1% sample of users.",
    "requirements": [
      "Add user_aisle_pacing_ratio to the Dataform pipeline in the three layers the repo uses: a silver building block, the skeleton forward-fill, and the final featu

## Clean up

A run whose patch applied pushed `agent/<task_id>`. Delete it from a terminal when you are done looking:

```powershell
git push origin --delete agent/<task_id>
```

The task document deletes itself 30 days after it completed.